In [1]:
import sys
import os
import math
import numpy as np

states = { "s": 0, "E": 1, "5": 2, "I" : 3, "e": 4}
id2state = {0: "s", 1: "E", 2: "5", 3: "I", 4: "e"}

state_transition_prob = np.array([
 [0.0, 1.0, 0.0, 0.0, 0.0], 
 [0.0, 0.9, 0.1, 0.0, 0.0], 
 [0.0, 0.0, 0.0, 1.0, 0.0],
 [0.0, 0.0, 0.0, 0.9, 0.1],
 [0.0, 0.0, 0.0, 0.0, 0.0]
])

emission_nuc_codes = {'A': 0, 'C': 1, 'G': 2, 'T': 3}

emission_probs = np.array([
 [0.00, 0.00, 0.00, 0.00], 
 [0.25, 0.25, 0.25, 0.25],
 [0.05, 0.00, 0.95, 0.00],
 [0.40, 0.10, 0.10, 0.40],
 [0.00, 0.00, 0.00, 0.00]
])

query_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"

In [2]:
def get_log_prob_for_state_path (state_path, query_sequence):
    res = math.log(0.25)
    for i in range(1, len(state_path)):
        res += math.log(
            state_transition_prob[ states[state_path[i-1]] ][ states[state_path[i]] ] *
            emission_probs[ states[state_path[i]] ][ emission_nuc_codes[query_sequence[i]] ]
        )
    return res

In [5]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEE5IIIIIIIIIIIIIIIIIII
k1 = get_log_prob_for_state_path("EEEEEE5IIIIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") +  math.log (0.1)
print (k1)

-43.89740030179307


In [4]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEE5IIIIIIIIIIIIIIIII
k2 = get_log_prob_for_state_path("EEEEEEEE5IIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k2)

-43.45111319916465


In [6]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEE5IIIIIIIIIIIII
k3 = get_log_prob_for_state_path("EEEEEEEEEEEE5IIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k3)

-43.944833355027704


In [7]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEE5IIIIIIIIII
k4 = get_log_prob_for_state_path("EEEEEEEEEEEEEEE5IIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k4)

-42.58225552052512


In [8]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEEEEE5IIIIIII
k5 = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEE5IIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k5)

-41.21967768602254


In [9]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEEEEEEEEE5III
k6 = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEE5III", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k6)

-41.713397841885595


In [10]:
# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEEEEEEEEEEEEEEEEEEEEEE
only_E = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEEEEEE", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (only_E)

-40.98025137355685


### Design of the Viterbi Value matrix

Rows correspond to the hidden states, and columns correspond to the emissions (observed nucleotide sequence).

Here I am showing the calculation for the first two nucleotides.

```
             C                                                          T     T
s [s-s-C(0.00) max(s-s-C-s-T, s-E-C-s-T, s-5-C-s-T, s-I-C-s-T, s-e-C-s-T)     .] 
E [s-E-C(0.25) max(s-s-C-E-T, s-E-C-E-T, s-5-C-E-T, s-I-C-E-T, s-e-C-E-T)     .] 
5 [s-5-C(0.00) max(s-s-C-5-T, s-E-C-5-T, s-5-C-5-T, s-I-C-5-T, s-e-C-5-T)     .]
I [s-I-C(0.00) max(s-s-C-I-T, s-E-C-I-T, s-5-C-I-T, s-I-C-I-T, s-e-C-I-T)     .]
e [s-e-C(0.00) max(s-s-C-e-T, s-E-C-e-T, s-5-C-e-T, s-I-C-e-T, s-e-C-e-T)     .]
```

It is important to remember that all computations are performed in log scale.

In [11]:
n_states = len(states)
T = len(query_sequence)

viterbi_value_matrix = np.full((n_states, T), -np.inf)
viterbi_trace_matrix = np.zeros((n_states, T), dtype=int)

In [12]:
first_obs = emission_nuc_codes[query_sequence[0]]

for s in range(n_states):
    trans_prob = state_transition_prob[states["s"]][s]
    emit_prob = emission_probs[s][first_obs]

    if trans_prob > 0 and emit_prob > 0:
        viterbi_value_matrix[s][0] = math.log(trans_prob) + math.log(emit_prob)
    else:
        viterbi_value_matrix[s][0] = -np.inf

    viterbi_trace_matrix[s][0] = states["s"]

In [13]:
def calculate_prob_for_a_node(prev_column, current_state, obs):
    max_val = -np.inf
    max_index = 0

    for prev_state in range(n_states):
        trans_prob = state_transition_prob[prev_state][current_state]
        emit_prob = emission_probs[current_state][obs]

        if trans_prob == 0 or emit_prob == 0:
            continue

        val = prev_column[prev_state] + math.log(trans_prob) + math.log(emit_prob)

        if val > max_val:
            max_val = val
            max_index = prev_state

    return max_val, max_index

In [14]:
for t in range(1, T):
    obs = emission_nuc_codes[query_sequence[t]]

    for s in range(n_states):
        val, prev_state = calculate_prob_for_a_node(
            viterbi_value_matrix[:, t-1],
            s,
            obs
        )

        viterbi_value_matrix[s][t] = val
        viterbi_trace_matrix[s][t] = prev_state

In [15]:
def traceback_path(viterbi_value_matrix, viterbi_trace_matrix):
    T = viterbi_value_matrix.shape[1]

    last_state = np.argmax(viterbi_value_matrix[:, T-1])

    path = [last_state]

    for t in range(T-1, 0, -1):
        last_state = viterbi_trace_matrix[last_state][t]
        path.insert(0, last_state)

    return path

In [16]:
path_indices = traceback_path(viterbi_value_matrix, viterbi_trace_matrix)

best_path = "".join([id2state[i] for i in path_indices])

print("Best state path:")
print(best_path)

Best state path:
EEEEEEEEEEEEEEEEEEEEEEEEEE


In [17]:
print("\nViterbi Value Matrix:")
print(viterbi_value_matrix)

print("\nViterbi Trace Matrix:")
print(viterbi_trace_matrix)


Viterbi Value Matrix:
[[        -inf         -inf         -inf         -inf         -inf
          -inf         -inf         -inf         -inf         -inf
          -inf         -inf         -inf         -inf         -inf
          -inf         -inf         -inf         -inf         -inf
          -inf         -inf         -inf         -inf         -inf
          -inf]
 [ -1.38629436  -2.87794924  -4.36960411  -5.86125899  -7.35291387
   -8.84456875 -10.33622362 -11.8278785  -13.31953338 -14.81118825
  -16.30284313 -17.79449801 -19.28615288 -20.77780776 -22.26946264
  -23.76111751 -25.25277239 -26.74442727 -28.23608214 -29.72773702
  -31.2193919  -32.71104677 -34.20270165 -35.69435653 -37.1860114
  -38.67766628]
 [        -inf         -inf         -inf         -inf -11.15957636
          -inf -11.19844713         -inf -14.18175689 -18.61785074
  -20.10950562 -21.6011605  -20.14837639         -inf -26.07612513
  -24.62334102 -29.05943488         -inf -29.09830565         -inf
  -35.02